# YOLO26 Object Detection on BHB signals in Simulated LISA Data

In [ ]:
from ultralytics import YOLO
from huggingface_hub import hf_hub_download
from yolo26mlx import YOLO
#!pip install safetensors

## Training

In [ ]:
# 1. Download the native MLX pretrained weights for the Medium model.
# MLX uses pure Numpy zip files (.npz) instead of PyTorch (.pt) files!
print("Downloading MLX weights...")
weights_path = hf_hub_download(repo_id="webAI-Official/yolo26l-mlx", filename="yolo26l.npz")

# 2. Initialize the pure-MLX YOLO model
print("Initializing MLX Model...")
model = YOLO(weights_path)

# 3. Execute training
# Notice we completely removed `device="mps"` because MLX natively 
# uses Apple Silicon without needing PyTorch device flags!
print("Starting ultra-fast MLX training...")
results = model.train(
    data="imageData5/BHBs.yaml",
    epochs=100,                        
    
    # --- MLX Speed Optimizations ---
    imgsz=640,          # Dropped from 1024 for speed; the compression was fine
    batch=4,
    
    # --- File Saving & Management ---
    project="LISA_YOLO26_MLX",   
    name="yolo26m_mlx_run"
)

print("MLX Training finished completely!")

## Evaluation + Prediction

In [ ]:
# Made by me, edited by Claude
# Claude could genuinely be a junior SWE

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms
from matplotlib.patches import Rectangle
from PIL import Image
from shapely.geometry import box as shp_box
from shapely.affinity import translate
from yolo26mlx import YOLO

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}


def _smooth(y, f=0.05):
    """Box filter smoothing (Ultralytics uses this on curves)."""
    nf = round(len(y) * f * 2) // 2 + 1
    p = np.ones(nf // 2)
    yp = np.concatenate((p * y[0], y, p * y[-1]), 0)
    return np.convolve(yp, np.ones(nf) / nf, mode="valid")


def _compute_ap(recall, precision):
    """AP via 101-point interpolation (COCO / Ultralytics convention)."""
    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([1.0], precision, [0.0]))
    mpre = np.flip(np.maximum.accumulate(np.flip(mpre)))          # envelope
    x = np.linspace(0, 1, 101)
    yi = np.interp(x, mrec, mpre)
    # np.trapz was renamed to np.trapezoid in NumPy 2.0
    _trap = getattr(np, "trapezoid", None) or np.trapz
    ap = _trap(yi, x)
    return ap, mpre, mrec


def _compute_metrics(all_conf, all_tp, n_gt, cm_conf=None, eps=1e-16):
    """Ultralytics-style curve arrays.

    Args:
        all_conf: 1D array of every prediction's confidence (all images pooled).
        all_tp:   1D bool array, True if that prediction is a TP @ iou_thr.
        n_gt:     total number of ground-truth boxes across all images.
        cm_conf:  confidence threshold to use for the confusion matrix counts.
                  If None, the best-F1 confidence is used.

    Returns dict with confidence grid + P, R, F1 curves, PR curve, AP, and
    confusion counts at the chosen threshold.
    """
    order = np.argsort(-all_conf)
    conf = all_conf[order]
    tp = all_tp[order].astype(float)
    fp = 1.0 - tp

    tpc = np.cumsum(tp)
    fpc = np.cumsum(fp)

    recall = tpc / (n_gt + eps)
    precision = tpc / (tpc + fpc + eps)

    x = np.linspace(0, 1, 1000)
    r_curve = np.interp(-x, -conf, recall, left=0)
    p_curve = np.interp(-x, -conf, precision, left=1)
    f1_curve = 2 * p_curve * r_curve / (p_curve + r_curve + eps)

    ap, mpre, mrec = _compute_ap(recall, precision)

    # best-F1 confidence (from smoothed curve, Ultralytics convention)
    best_i = _smooth(f1_curve).argmax()
    best_conf = float(x[best_i])

    # choose the confusion-matrix threshold
    used_best = cm_conf is None
    cm_thresh = best_conf if used_best else float(cm_conf)

    tp_at = int(tp[conf >= cm_thresh].sum())
    fp_at = int(fp[conf >= cm_thresh].sum())
    fn_at = int(n_gt - tp_at)

    return dict(x=x, p=p_curve, r=r_curve, f1=f1_curve,
                pr_p=mpre, pr_r=mrec, ap=ap,
                best_conf=best_conf, cm_thresh=cm_thresh, cm_used_best=used_best,
                tp=tp_at, fp=fp_at, fn=fn_at)


def _match_for_metrics(preds, gt_boxes, iou_thr=0.5):
    """Return (conf_list, tp_list) for a single image (preds sorted by conf desc)."""
    preds = sorted(preds, key=lambda p: -p[4])
    taken = [False] * len(gt_boxes)
    confs, tps = [], []
    for (x1, y1, x2, y2, c) in preds:
        pg = shp_box(x1, y1, x2, y2)
        best_iou, best_j = 0.0, -1
        for j, g in enumerate(gt_boxes):
            if taken[j]:
                continue
            u = g.union(pg).area
            iou = g.intersection(pg).area / u if u > 0 else 0.0
            if iou > best_iou:
                best_iou, best_j = iou, j
        is_tp = best_j >= 0 and best_iou >= iou_thr
        if is_tp:
            taken[best_j] = True
        confs.append(c)
        tps.append(is_tp)
    return confs, tps


def nms_indices(xyxy, scores, iou_thresh=0.9):
    """Greedy NMS. Returns indices of boxes to KEEP, highest score first.

    Args:
        xyxy:   (N, 4) array of [x1, y1, x2, y2].
        scores: (N,) array of confidences.
        iou_thresh: IoU above which two boxes are considered the same object.

    Returns:
        list[int] of kept indices.
    """
    xyxy = np.asarray(xyxy, dtype=float).reshape(-1, 4)
    scores = np.asarray(scores, dtype=float).reshape(-1)
    if len(scores) == 0:
        return []

    x1, y1, x2, y2 = xyxy[:, 0], xyxy[:, 1], xyxy[:, 2], xyxy[:, 3]
    areas = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    order = scores.argsort()[::-1]   # highest confidence first

    keep = []
    while order.size > 0:
        i = order[0]
        keep.append(int(i))
        if order.size == 1:
            break
        rest = order[1:]
        # intersection of box i with all remaining
        xx1 = np.maximum(x1[i], x1[rest])
        yy1 = np.maximum(y1[i], y1[rest])
        xx2 = np.minimum(x2[i], x2[rest])
        yy2 = np.minimum(y2[i], y2[rest])
        w = np.clip(xx2 - xx1, 0, None)
        h = np.clip(yy2 - yy1, 0, None)
        inter = w * h
        union = areas[i] + areas[rest] - inter
        iou = np.divide(inter, union, out=np.zeros_like(inter), where=union > 0)
        # drop boxes that overlap too much with i (they're duplicates)
        order = rest[iou <= iou_thresh]
    return keep


def apply_nms(result, iou_thresh=0.9):
    """Run NMS on a YOLO26 predict() Results object; return a filtered copy.

    Works generally on YOLO26 results (not just for this visualization).
    Falls back to returning the raw kept (xyxy, conf) if the Results object
    can't be safely sliced in-place.

    Args:
        result: a single Results object from model.predict(...)[i].
        iou_thresh: IoU threshold for treating two boxes as the same object.

    Returns:
        (result_like, kept_indices) where result_like is either the filtered
        Results object (if sliceable) or the original result, and kept_indices
        is the list of surviving box indices.
    """
    b = result.boxes
    if b is None or len(b) == 0:
        return result, []

    xyxy = np.array(b.xyxy).reshape(-1, 4)
    conf = np.array(b.conf).reshape(-1)
    keep = nms_indices(xyxy, conf, iou_thresh=iou_thresh)

    # Try to slice the Boxes object so downstream code still gets a Results-like obj.
    try:
        result.boxes = b[keep]     # many ports support integer-list indexing
    except Exception:
        pass                        # leave result unchanged; caller can use keep

    return result, keep


def _preds_from_result(result, use_nms=False, nms_iou=0.9):
    """Return [(x1,y1,x2,y2,conf), ...], optionally after NMS."""
    b = result.boxes
    if b is None or len(b) == 0:
        return []
    xyxy = np.array(b.xyxy).reshape(-1, 4)
    conf = np.array(b.conf).reshape(-1)
    idx = range(len(conf))
    if use_nms:
        idx = nms_indices(xyxy, conf, iou_thresh=nms_iou)
    return [(float(xyxy[i, 0]), float(xyxy[i, 1]), float(xyxy[i, 2]),
             float(xyxy[i, 3]), float(conf[i])) for i in idx]


def _plot_results(m, out_dir, cls_name="BHB"):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    x = m["x"]

    def _curve(xv, yv, xl, yl, fname, best=None):
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.plot(xv, yv, linewidth=2, label=cls_name)
        if best is not None:
            ax.axvline(best, ls="--", color="gray",
                       label=f"best conf={best:.3f}")
        ax.set_xlabel(xl); ax.set_ylabel(yl); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
        ax.set_title(f"{yl} vs {xl}"); ax.legend(); ax.grid(alpha=0.3)
        fig.savefig(out_dir / fname, dpi=150, bbox_inches="tight")
        plt.show(); plt.close(fig)

    _curve(x, m["p"],  "Confidence", "Precision", "P_curve.png", m["best_conf"])
    _curve(x, m["r"],  "Confidence", "Recall",    "R_curve.png", m["best_conf"])
    _curve(x, m["f1"], "Confidence", "F1",        "F1_curve.png", m["best_conf"])

    # PR curve
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(m["pr_r"], m["pr_p"], linewidth=2, label=f"{cls_name} AP={m['ap']:.3f}")
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_title("Precision-Recall Curve"); ax.legend(); ax.grid(alpha=0.3)
    fig.savefig(out_dir / "PR_curve.png", dpi=150, bbox_inches="tight")
    plt.show(); plt.close(fig)

    # confusion matrix (rows=predicted, cols=actual): [BHB, background]
    tp, fp, fn = m["tp"], m["fp"], m["fn"]
    cm = np.array([[tp, fp],
                   [fn, 0]], dtype=float)
    labels = [cls_name, "background"]

    # threshold annotation text
    if m["cm_used_best"]:
        thr_text = f"conf threshold = {m['cm_thresh']:.3f}  (best F1)"
    else:
        thr_text = f"conf threshold = {m['cm_thresh']:.3f}  (user-specified)"

    for norm in (False, True):
        mat = cm.copy()
        if norm:
            col_sum = mat.sum(axis=0, keepdims=True)
            mat = np.divide(mat, col_sum, out=np.zeros_like(mat), where=col_sum != 0)
        fig, ax = plt.subplots(figsize=(6, 5.4))
        im = ax.imshow(mat, cmap="Blues")
        ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
        ax.set_xticklabels(labels); ax.set_yticklabels(labels)
        ax.set_xlabel("Actual"); ax.set_ylabel("Predicted")
        ax.set_title("Confusion Matrix" + (" (Normalized)" if norm else ""))
        for i in range(2):
            for j in range(2):
                ax.text(j, i, f"{mat[i, j]:.2f}" if norm else f"{int(mat[i, j])}",
                        ha="center", va="center",
                        color="white" if mat[i, j] > mat.max() / 2 else "black")
        fig.colorbar(im)
        # threshold caption below the axes so it never overlaps cells/labels
        fig.text(0.5, 0.02, thr_text, ha="center", va="bottom",
                 fontsize=10, fontweight="bold")
        fig.subplots_adjust(bottom=0.18)   # reserve room for the caption
        fname = "confusion_matrix_normalized.png" if norm else "confusion_matrix.png"
        fig.savefig(out_dir / fname, dpi=150, bbox_inches="tight")
        plt.show(); plt.close(fig)


def _ensure_model(model):
    """Accept a YOLO instance OR a weights path; relabel to single BHB class."""
    m = YOLO(str(model)) if isinstance(model, (str, Path)) else model
    m.nc = 1
    m.names = {0: "BHB"}
    if getattr(m, "model", None) is not None:
        try:
            m.model.nc = 1
            m.model.names = {0: "BHB"}
        except Exception:
            pass
    return m


def _gather_images(path):
    """Return sorted image paths from a file or (recursively) a folder."""
    path = Path(path)
    if path.is_file():
        return [path] if path.suffix.lower() in IMG_EXTS else []
    if path.is_dir():
        return sorted(p for p in path.rglob("*") if p.suffix.lower() in IMG_EXTS)
    raise FileNotFoundError(f"Path not found: {path}")


def _find_label(img_path):
    """Map .../images/<split>/name.jpg -> .../labels/<split>/name.txt if it exists."""
    parts = list(img_path.parts)
    if "images" in parts:
        idx = len(parts) - 1 - parts[::-1].index("images")   # last 'images'
        parts[idx] = "labels"
        label = Path(*parts).with_suffix(".txt")
        return label if label.exists() else None
    return None


def _extract_preds(result):
    """Return [(x1,y1,x2,y2,conf), ...] from a Results object (robust to this MLX port)."""
    b = result.boxes
    if b is None or len(b) == 0:
        return []
    xyxy = np.array(b.xyxy).reshape(-1, 4)
    conf = np.array(b.conf).reshape(-1)
    return [(float(xyxy[i, 0]), float(xyxy[i, 1]), float(xyxy[i, 2]),
             float(xyxy[i, 3]), float(conf[i])) for i in range(xyxy.shape[0])]


def _load_gt(label_path, img_w, img_h):
    """Read YOLO-format labels -> list of shapely boxes in pixel coords."""
    boxes = []
    for line in Path(label_path).read_text().splitlines():
        parts = line.strip().split()
        if not parts:
            continue
        _, xc, yc, bw, bh = parts[:5]
        xc, yc, bw, bh = float(xc), float(yc), float(bw), float(bh)
        boxes.append(shp_box((xc - bw / 2) * img_w, (yc - bh / 2) * img_h,
                             (xc + bw / 2) * img_w, (yc + bh / 2) * img_h))
    return boxes


def _conf_color(c):
    if c is None:      return "white"
    if c < 0.365:      return "red"
    if c < 0.5:        return "yellow"
    if c < 0.8:        return "lightgreen"
    return "darkgreen"


def _iou_color(v):
    if v is None or v < 0.3: return "red"
    if v < 0.5:              return "yellow"
    if v < 0.75:            return "lightgreen"
    return "darkgreen"


def _match(preds, gt_boxes, iou_thr=0.5):
    """Greedy match preds (sorted by conf) to GT.

    Returns:
        rows: list of dicts {pred, conf, actual, iou} for every drawn item
              (includes false negatives with pred=None).
    """
    taken = [False] * len(gt_boxes)
    rows = []
    for (x1, y1, x2, y2, conf) in preds:
        pgeom = shp_box(x1, y1, x2, y2)
        best_iou, best_j = 0.0, -1
        for j, g in enumerate(gt_boxes):
            if taken[j]:
                continue
            inter = g.intersection(pgeom).area
            union = g.union(pgeom).area
            iou = inter / union if union > 0 else 0.0
            if iou > best_iou:
                best_iou, best_j = iou, j
        if best_j >= 0 and best_iou > iou_thr:
            taken[best_j] = True
            rows.append({"pred": pgeom, "conf": conf,
                         "actual": gt_boxes[best_j], "iou": best_iou})
        else:
            rows.append({"pred": pgeom, "conf": conf, "actual": None, "iou": None})
    # false negatives
    for j, g in enumerate(gt_boxes):
        if not taken[j]:
            rows.append({"pred": None, "conf": None, "actual": g, "iou": None})
    return rows


def _repel_labels(ax, fig, anchors, texts, colors, box_face="blue"):
    """Place each label just ABOVE its box (flush with the box's left edge).
    Only push a label upward when it would actually overlap another label
    (i.e. their horizontal ranges intersect). Leader lines keep the
    box<->label association clear.

    Args:
        anchors: list of (x, y) data-coords = box TOP-LEFT.
        texts:   list of strings (may be multi-line).
        colors:  list of text colors.
    """
    if not anchors:
        return

    # va="bottom": stored y = label's BOTTOM edge; text grows upward.
    txt_objs = []
    for (x, y), s, col in zip(anchors, texts, colors):
        t = ax.annotate(
            s, xy=(x, y), xytext=(x, y), textcoords="data",
            color=col, fontsize=9, fontweight="bold",
            bbox=dict(facecolor=box_face, alpha=0.75, edgecolor="none", pad=1.0),
            va="bottom", ha="left", clip_on=False,
            arrowprops=dict(arrowstyle="-", color=box_face, lw=0.8),
        )
        txt_objs.append(t)

    # measure label width + height in DATA units (render once)
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    inv = ax.transData.inverted()
    widths, heights = [], []
    for t in txt_objs:
        ext = t.get_window_extent(renderer)
        (x0d, y0d) = inv.transform((ext.x0, ext.y0))
        (x1d, y1d) = inv.transform((ext.x1, ext.y1))
        widths.append(abs(x1d - x0d))
        heights.append(abs(y1d - y0d))

    gap = max(heights) * 0.12 if heights else 3.0

    # process from bottom of image (largest y) to top; imshow y grows DOWNWARD
    order = sorted(range(len(anchors)), key=lambda i: anchors[i][1], reverse=True)

    placed = []  # (x_left, x_right, top_y, bottom_y) of labels already positioned
    for i in order:
        x, y = anchors[i]
        w, h = widths[i], heights[i]
        bottom = y - gap                 # ideal: just above the box top
        x_left, x_right = x, x + w

        # push up only past labels we actually overlap horizontally
        changed = True
        while changed:
            changed = False
            for (px_l, px_r, p_top, p_bot) in placed:
                x_overlap = (x_left < px_r) and (x_right > px_l)
                top = bottom - h
                y_overlap = (top < p_bot) and (bottom > p_top)
                if x_overlap and y_overlap:
                    bottom = p_top - gap   # move just above the conflicting label
                    changed = True

        txt_objs[i].set_position((x, bottom))
        placed.append((x_left, x_right, bottom - h, bottom))


# ---------------------------------------------------------------
# CROPPED view: one subplot per detection/GT box
# ---------------------------------------------------------------
def _plot_cropped(img, rows, title, n_det, n_act, save_path=None):
    W, H = img.size
    arr = np.array(img)
    cols = 4
    n = len(rows)
    if n == 0:
        return
    r = int(np.ceil(n / cols))
    fig, ax = plt.subplots(r, cols, figsize=(cols * 6, r * 5))
    axf = np.atleast_1d(ax).ravel()
    added_pred = added_act = False

    for i, row in enumerate(rows):
        pred, actual = row["pred"], row["actual"]
        pe, ae = pred is not None, actual is not None
        pb = pred.bounds if pe else (-1, -1, -1, -1)
        ab = actual.bounds if ae else (-1, -1, -1, -1)
        
        if pe and ae:
            cx0, cy0 = min(ab[0], pb[0]), min(ab[1], pb[1])
            cx1, cy1 = max(ab[2], pb[2]), max(ab[3], pb[3])
        elif pe:
            cx0, cy0, cx1, cy1 = pb
        else:
            cx0, cy0, cx1, cy1 = ab
        
        mnX = int(np.clip(cx0 - 15, 0, None)); mnY = int(np.clip(cy0 - 35, 0, None))
        mxX = int(np.clip(cx1 + 15, None, W - 1)); mxY = int(np.clip(cy1 + 15, None, H - 1))
        axf[i].imshow(arr[mnY:mxY + 1, mnX:mxX + 1])
        
        if pe:
            sp = translate(pred, xoff=-mnX, yoff=-mnY)
            xs, ys = sp.exterior.xy
            lbl = "Predicted" if not added_pred else None
            added_pred = True
            axf[i].add_patch(Rectangle((min(xs), min(ys)), max(xs) - min(xs),
                                       max(ys) - min(ys), linewidth=1.2,
                                       edgecolor="blue", facecolor="none", label=lbl))
            c, iou = row["conf"], row["iou"]
            lx, ly = min(xs), min(ys)
            td = axf[i].transData
            t_iou = mtransforms.offset_copy(td, fig=fig, x=0, y=4, units="points")
            t_conf = mtransforms.offset_copy(td, fig=fig, x=0, y=18, units="points")
            axf[i].text(lx, ly, f"Conf: {c:.2f}", transform=t_conf,
                        color=_conf_color(c), fontsize=10, fontweight="bold",
                        bbox=dict(facecolor="blue", alpha=0.75, edgecolor="none", pad=1.0),
                        va="bottom", ha="left", clip_on=False)
            axf[i].text(lx, ly, f"IoU: {iou:.2f}" if iou is not None else "IoU: N/A",
                        transform=t_iou, color=_iou_color(iou), fontsize=10, fontweight="bold",
                        bbox=dict(facecolor="blue", alpha=0.75, edgecolor="none", pad=1.0),
                        va="bottom", ha="left", clip_on=False)
        if ae:
            sa = translate(actual, xoff=-mnX, yoff=-mnY)
            xs, ys = sa.exterior.xy
            lbl = "Actual" if not added_act else None
            added_act = True
            axf[i].add_patch(Rectangle((min(xs), min(ys)), max(xs) - min(xs),
                                       max(ys) - min(ys), linewidth=1.2,
                                       edgecolor="lime", facecolor="none",
                                       linestyle="dashed", label=lbl))
        axf[i].set_axis_off()
    
    for d in range(n, len(axf)):
        fig.delaxes(axf[d])
    
    fig.text(0.02, 1.02, f"Detected Boxes: {n_det}\nActual Boxes: {n_act}",
             fontsize=16, va="top", ha="left", fontweight="bold")
    ll = [a.get_legend_handles_labels() for a in fig.axes]
    lines, labels = [sum(x, []) for x in zip(*ll)] if ll else ([], [])
    if lines:
        fig.legend(lines, labels, fontsize=16, loc="upper center",
                   ncol=2, bbox_to_anchor=(0.5, 1.02))
    fig.suptitle(title, y=1.06, fontsize=13)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    if save_path is not None:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)


# ---------------------------------------------------------------
# FULL view: whole image with every box overlaid
# ---------------------------------------------------------------
def _plot_full(img, rows, title, n_det, n_act, save_path=None):
    fig, ax = plt.subplots(figsize=(12, 12))
    ax.imshow(np.array(img))
    added_pred = added_act = False
    
    anchors, texts, colors = [], [], []
    for row in rows:
        if row["pred"] is not None:
            x0, y0, x1, y1 = row["pred"].bounds
            lbl = "Predicted" if not added_pred else None
            added_pred = True
            ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, linewidth=1.2,
                                   edgecolor="blue", facecolor="none", label=lbl))
            c, iou = row["conf"], row["iou"]
            iou_str = f"IoU: {iou:.2f}" if iou is not None else "IoU: N/A"
            anchors.append((x0, y0))
            texts.append(f"Conf: {c:.2f}\n{iou_str}")
            colors.append(_conf_color(c))
        if row["actual"] is not None:
            x0, y0, x1, y1 = row["actual"].bounds
            lbl = "Actual" if not added_act else None
            added_act = True
            ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, linewidth=1.2,
                                   edgecolor="lime", facecolor="none",
                                   linestyle="dashed", label=lbl))
    
    if anchors:
        _repel_labels(ax, fig, anchors, texts, colors)
    
    ax.set_axis_off()
    ax.set_title(f"{title}   |   Detected: {n_det}  Actual: {n_act}", fontsize=12)
    if added_pred or added_act:
        ax.legend(loc="upper right", fontsize=11)
    plt.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)


# ---------------------------------------------------------------
# MAIN ENTRY POINT
# ---------------------------------------------------------------
def visualize(model, source, crop=False, visualConf=0.25, imgsz=640, iou_thr=0.5,
              save_dir=None, generateResultsPlots=False, metrics_conf=0.001,
              NMS=False, nms_iou=0.9, confusionMatrixConf=None):
    """Predict + visualize, optionally saving images and results plots.

    Args:
        model:    a YOLO instance OR a path to weights.
        source:   single image, or any folder level (root, .../images, or split).
        crop:     True -> zoomed subplot per box; False -> full image overlay.
        conf:     confidence threshold for the VISUALIZED predictions.
        imgsz:    inference image size.
        iou_thr:  IoU threshold for matching predictions to ground truth.
        save_dir: parent folder. Images -> save_dir/images,
                  results plots -> save_dir/resultsPlots. None = don't save.
        generateResultsPlots: if True, compute + save the 6 Ultralytics-style plots
                  (needs ground-truth labels). Uses metrics_conf for the sweep.
        metrics_conf: low conf threshold used when gathering predictions for metrics
                  (Ultralytics uses 0.001). Visualization still uses `conf`.
        NMS:      if True, apply NMS to predictions before visualizing AND before
                  computing metrics, so results reflect the model WITH NMS added.
                  If False, raw (non-NMS) predictions are used everywhere.
        nms_iou:  IoU threshold for NMS (two boxes above this = same object).
        visualConf: confidence threshold for the VISUALIZED predictions (the
                    boxes drawn on the images). Independent of the metric sweep.
        metrics_conf: low confidence floor for gathering predictions used in the
                    P/R/F1/PR sweep (Ultralytics uses 0.001).
        confusionMatrixConf: confidence threshold used specifically for the
                    confusion matrices. If None (default), the best-F1
                    confidence from the sweep is used.
    """
    m = _ensure_model(model)
    images = _gather_images(source)
    if not images:
        print(f"No images found at: {source}")
        return
    
    img_dir = plot_dir = None
    if save_dir is not None:
        img_dir = Path(save_dir) / "images"
        img_dir.mkdir(parents=True, exist_ok=True)
        plot_dir = Path(save_dir) / "resultsPlots"
    
    print(f"Found {len(images)} image(s). Running predictions"
          f"{' with NMS' if NMS else ''}...\n")
    all_conf, all_tp, n_gt_total = [], [], 0
    
    for img_path in images:
        # --- visualization predictions (visualConf) ---
        result = m.predict(str(img_path), conf=visualConf, imgsz=imgsz, save=False)[0]
        preds = _preds_from_result(result, use_nms=NMS, nms_iou=nms_iou)

        img = Image.open(img_path).convert("RGB")
        W, H = img.size
        label_path = _find_label(img_path)
        
        if label_path is not None:
            gt_boxes = _load_gt(label_path, W, H)
            rows = _match(preds, gt_boxes, iou_thr=iou_thr)
            n_act = len(gt_boxes)
        else:
            gt_boxes = []
            rows = [{"pred": shp_box(x1, y1, x2, y2), "conf": c,
                     "actual": None, "iou": None}
                    for (x1, y1, x2, y2, c) in preds]
            n_act = 0
        
        # --- metrics gathering (low-conf sweep, needs labels) ---
        if generateResultsPlots and label_path is not None:
            lowres = m.predict(str(img_path), conf=metrics_conf,
                               imgsz=imgsz, save=False)[0]
            lowpreds = _preds_from_result(lowres, use_nms=NMS, nms_iou=nms_iou)
            confs, tps = _match_for_metrics(lowpreds, gt_boxes, iou_thr=iou_thr)
            all_conf.extend(confs)
            all_tp.extend(tps)
            n_gt_total += len(gt_boxes)
        
        save_path = None
        if img_dir is not None:
            save_path = img_dir / f"{img_path.parent.name}_{img_path.stem}.png"
    
        title = img_path.name
        if crop:
            _plot_cropped(img, rows, title, len(preds), n_act, save_path=save_path)
        else:
            _plot_full(img, rows, title, len(preds), n_act, save_path=save_path)
    
    # --- results plots after all images ---
    if generateResultsPlots:
        if not all_conf or n_gt_total == 0:
            print("Cannot generate results plots: no labeled ground truth found "
                  "for the given source (need matching .txt label files).")
            return
        metrics = _compute_metrics(np.array(all_conf, dtype=float),
                                   np.array(all_tp, dtype=bool),
                                   n_gt_total,
                                   cm_conf=confusionMatrixConf)
        target = plot_dir if plot_dir is not None else Path("resultsPlots")
        _plot_results(metrics, target, cls_name=m.names.get(0, "BHB"))
        print(f"\nResults plots saved to: {target}")
        print(f"  AP@{iou_thr}: {metrics['ap']:.4f}")
        print(f"  Best F1 confidence: {metrics['best_conf']:.3f}")
        cm_kind = "best-F1" if metrics["cm_used_best"] else "user-specified"
        print(f"  Confusion matrix @ conf={metrics['cm_thresh']:.3f} ({cm_kind})"
              f"  ->  TP={metrics['tp']}  FP={metrics['fp']}  FN={metrics['fn']}")



In [ ]:
weights = "LISA_YOLO26_MLX/yolo26l_mlx_run/yolo26l_best.safetensors"

# Visualize + save annotated images + generate all 6 metric plots
visualize(
    weights,
    "imageData5/images/test",
    crop=False,
    save_dir="LISA_YOLO26_MLX/yolo26l_mlx_run/myResults",          # -> myResults/images + myResults/resultsPlots
    generateResultsPlots=True,
    NMS = True,
    nms_iou = 0.60,
    confusionMatrixConf = None,
    visualConf = 0.179
)

In [ ]:
visualize(weights, "sgramTest.jpg", crop = False, save_dir = 'LISA_YOLO26_MLX/yolo26l_mlx_run/mojitoTest', generateResultsPlots = False, NMS = True, nms_iou = 0.45, visualConf = 0.179)

